In [1]:
import pandas as pd
import numpy as np

from loader import cargar_dataset

In [2]:
PATH_VAL_CLIM = 'Valores_Climatologicos_1970_2024_con_Coordenadas.csv'

In [3]:
df = cargar_dataset(url = PATH_VAL_CLIM, variables_brutas = True)

In [4]:
display(df.sample(20))

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,tmax,velmedia,sol,presMax,presMin,hrMedia,dir,racha,hrMax,hrMin,lon,lat
5883106,2020-02-25,C649R,"TELDE, MELENARA",LAS PALMAS,9,"21,6","0,0","15,6","27,6","2,5",NaN,NaN,NaN,48.0,35.0,"10,0",71.0,18.0,-15.377778,27.986667
2531433,2008-02-20,1387,A CORUÑA,A CORUÑA,57,"16,1","0,0","13,0","19,2","2,5","3,6","1015,1","1008,3",77.0,16.0,"7,5",NaN,NaN,-8.421389,43.365833
5590753,2019-03-20,5860E,"MOGUER, EL ARENOSILLO",HUELVA,41,"14,2","0,0","8,1","20,2","3,6",NaN,NaN,NaN,53.0,99.0,"7,8",70.0,36.0,-6.738056,37.098056
345572,1978-02-10,1521I,O PÁRAMO,LUGO,403,NaN,"0,0",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-7.499444,42.845278
2798963,2009-08-09,6058I,ESTEPONA,MALAGA,19,"24,3","0,0","17,0","31,6","1,1",NaN,"1012,5","1009,8",58.0,18.0,"7,5",82.0,27.0,-5.155,36.416944
3671836,2012-11-16,1012P,IRUN,GIPUZKOA,120,"17,7","0,0","13,8","21,6","0,6",NaN,NaN,NaN,62.0,18.0,"6,9",78.0,51.0,-1.796667,43.326389
7014718,2023-09-22,1012P,IRUN,GIPUZKOA,120,"15,8","11,4","14,1","17,5","2,8",NaN,NaN,NaN,79.0,28.0,"10,8",92.0,62.0,-1.796667,43.326389
1767276,2001-08-10,1658,FOLGOSO DO COUREL,LUGO,612,NaN,"0,0",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-7.191667,42.588611
3991730,2013-12-28,1041A,ZUMAIA,GIPUZKOA,28,"10,8","4,6","8,5","13,0","2,2",NaN,NaN,NaN,62.0,31.0,"19,7",95.0,48.0,-2.251111,43.302222
1621944,1999-10-31,8293X,XÀTIVA,VALENCIA,88,"19,8","0,0","11,4","28,1","0,8",NaN,NaN,NaN,NaN,24.0,"5,3",NaN,NaN,-0.523056,39.001667


### Preprocesado del Dataframe

In [5]:
df_prep = df.copy(deep = True)

In [6]:
COLS_VARIABLES = [
    'tmed',
    'prec',
    'tmin',
    'tmax',
    'velmedia',
    'sol',
    'presMax',
    'presMin',
    'hrMedia',
    'dir',
    'racha',
    'hrMax',
    'hrMin'
]

COL_PREC = 'prec'
COL_PROVINCIA = 'provincia'
COL_INDICATIVO = 'indicativo'
COL_TMAX = 'tmax'
COL_TMIN = 'tmin'
COL_DIR = 'dir'
COL_FECHA = 'fecha'

In [7]:
# Eliminamos las filas de precipitación acumulada, y cambiamos los valores 'Ip' por 0 (Ip = inferior a 0,1 mm)
df_prep = df_prep[df_prep[COL_PREC] != 'Acum']
df_prep[COL_PREC] = df_prep[COL_PREC].replace('Ip', 0)

In [8]:
# Reemplazamos las comas por puntos para poder cargar los números como decimales (floats)
df_prep[COLS_VARIABLES] = df_prep[COLS_VARIABLES].replace(',', '.', regex=True).apply(pd.to_numeric, errors = 'raise')

In [9]:
df_prep.dtypes

fecha         datetime64[ns]
indicativo    string[python]
nombre        string[python]
provincia     string[python]
altitud                Int64
tmed                 float64
prec                 float64
tmin                 float64
tmax                 float64
velmedia             float64
sol                  float64
presMax              float64
presMin              float64
hrMedia              float64
dir                  float64
racha                float64
hrMax                float64
hrMin                float64
lon                  Float64
lat                  Float64
dtype: object

In [10]:
# Corregimos inconsistencias en nombres de provincias
df_prep.loc[df_prep[COL_PROVINCIA] == 'BALEARES', COL_PROVINCIA] = 'ILLES BALEARS'
df_prep.loc[df_prep[COL_PROVINCIA] == 'STA. CRUZ DE TENERIFE', COL_PROVINCIA] = 'SANTA CRUZ DE TENERIFE'

In [11]:
# Borramos dos datos de una medición anómala
df_prep.loc[
    (df_prep[COL_INDICATIVO] == '6084X') &
    (df_prep[COL_TMIN] == 50.0) &
    (df_prep[COL_TMAX] == -50.0),
    [COL_TMIN, COL_TMAX]
] = np.nan

In [12]:
# Dejamos a NaN aquellos valores desconocidos (88: 'Desconocida') o múltiples (99: 'Varias), según el fichero de metadatos
df[COL_DIR] = df[COL_DIR].replace([99, 88], np.nan)

In [13]:
df_prep.sample(20)

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,tmax,velmedia,sol,presMax,presMin,hrMedia,dir,racha,hrMax,hrMin,lon,lat
2692648,2009-02-13,7080X,MORATALLA,MURCIA,955,7.4,0.0,1.6,13.3,NaN,NaN,NaN,NaN,47.0,NaN,NaN,83.0,34.0,-1.986389,38.268333
3969720,2013-11-30,3514B,TORNAVACAS,CACERES,991,5.4,0.0,1.8,8.9,6.4,NaN,NaN,NaN,54.0,7.0,21.1,73.0,22.0,-5.678333,40.258611
3046990,2010-07-31,4478X,BADAJOZ,BADAJOZ,174,29.2,0.0,20.4,38.1,2.5,NaN,NaN,NaN,40.0,24.0,8.3,76.0,18.0,-7.009444,38.886111
221322,1975-06-06,8175,ALBACETE BASE AÉREA,ALBACETE,702,17.7,0.0,10.0,25.4,3.3,4.5,938.9,937.7,54.0,NaN,NaN,NaN,NaN,-1.856389,38.954167
4516051,2015-09-25,1435C,NOIA,A CORUÑA,128,20.0,0.0,14.9,25.0,NaN,NaN,NaN,NaN,79.0,NaN,NaN,96.0,63.0,-8.876111,42.800278
7415341,2024-12-18,3260B,TOLEDO,TOLEDO,513,7.2,0.0,0.1,14.3,0.8,6.0,972.0,966.4,72.0,11.0,3.9,96.0,49.0,-4.045278,39.884722
3763697,2013-03-14,9898,"HUESCA, AEROPUERTO",HUESCA,546,4.6,0.0,-0.2,9.4,12.2,10.7,948.2,942.6,54.0,30.0,23.3,65.0,44.0,-0.325556,42.084444
45105,1971-03-20,1159,SAN VICENTE DE LA BARQUERA,CANTABRIA,38,NaN,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.392222,43.393333
3779207,2013-04-03,8300X,CARCAIXENT,VALENCIA,25,14.2,0.2,6.7,21.7,3.3,NaN,NaN,NaN,61.0,14.0,11.1,89.0,37.0,-0.445833,39.113333
6736698,2022-11-08,1484C,PONTEVEDRA,PONTEVEDRA,113,15.2,17.4,12.9,17.6,2.5,0.0,1001.4,996.1,88.0,19.0,12.2,99.0,80.0,-8.615833,42.438333


In [14]:
display(df_prep.describe().style.format({col: "{:.2f}" for col in df_prep.columns if col != COL_FECHA})) # Formateamos todo con 2 decimales excepto la fecha

,fecha,altitud,tmed,prec,tmin,tmax,velmedia,sol,presMax,presMin,hrMedia,dir,racha,hrMax,hrMin,lon,lat
count,7424501,7424501.00,7058645.00,7181646.00,7063148.00,7063427.00,5648244.00,2063944.00,2539482.00,2536756.00,6168239.00,5431040.00,5430969.00,4856022.00,4856424.00,7424501.00,7424501.00
mean,2008-12-07 07:51:23.198238208,489.96,15.25,1.73,9.93,20.57,2.94,7.06,970.75,966.46,65.76,24.25,9.82,87.48,47.58,-4.55,39.20
min,1970-01-01 00:00:00,1.00,-16.80,0.00,-25.40,-15.00,0.00,0.00,693.20,686.40,1.00,-9.00,0.00,1.00,0.00,-18.11,27.67
25%,2002-08-13 00:00:00,98.00,10.10,0.00,5.00,14.80,1.70,3.80,938.70,934.50,54.00,11.00,6.90,82.00,33.00,-6.33,37.96
50%,2013-01-10 00:00:00,430.00,15.20,0.00,10.10,20.30,2.50,7.80,985.10,980.10,68.00,22.00,9.20,92.00,47.00,-3.83,40.45
75%,2019-02-25 00:00:00,755.00,20.50,0.20,15.00,26.20,3.60,10.20,1013.60,1009.50,79.00,30.00,11.90,97.00,61.00,-1.28,42.24
max,2024-12-31 00:00:00,3092.00,40.40,710.80,37.20,47.60,44.40,15.30,1045.50,1042.00,100.00,99.00,68.90,111.00,100.00,4.32,43.79
std,nan,429.35,6.94,6.12,6.73,7.91,2.07,4.05,50.84,50.80,17.47,21.01,4.18,13.01,19.24,5.07,4.35


In [15]:
df_prep.to_csv('Valores_Climatologicos_1970_2024_Limpios.csv', sep = ';', index = False)